# Pandas Pivot Tables
### Topics Covered (in order)
1. **Setup & Data** — Load and inspect the dataset
2. **Categorical Data** — Control sort order in output
3. **Basic Pivot Table** — Single index
4. **Multiple Indices** — Drill deeper with hierarchical rows
5. **The `values` Parameter** — Choose what to aggregate
6. **Aggregation with `aggfunc`** — mean, sum, count, and more
7. **The `columns` Parameter** — Spread a category sideways
8. **Handling NaNs with `fill_value`** — Replace blanks with 0
9. **Moving Items Between Index and Columns** — Same data, different shape
10. **Margins (Totals)** — Add row and column totals
11. **Filtering a Pivot Table** — Slice results with `.query()`
12. **Styling** — Highlight, gradient, and conditional colours

---

## ⚙️ Step 0 — Install Required Libraries
Run this cell once before anything else.

In [ ]:
# Install pandas — the core library for data manipulation and pivot tables
!pip install pandas

# Install numpy — provides mathematical functions like np.sum, np.mean, np.std
!pip install numpy

---

---

# 📌 TOPIC 1 — Setup & Data

> Before building any pivot table, always load and inspect your data first.  
> You need to know your **column names**, **data types**, and **what each row represents**.

In [ ]:
# Import pandas with the standard alias 'pd'
import pandas as pd

# Import numpy with the standard alias 'np' — used for aggregate functions later
import numpy as np

print("Libraries loaded!")

In [ ]:
# Read the CSV file into a DataFrame — this is our sales pipeline data
df = pd.read_csv('data/sales.csv')

# Preview the first 5 rows to see what the data looks like
display(df.head())

In [ ]:
# Check the data type of each column — important before building pivots
df.dtypes

**What's in our data?**
- `Manager` — who manages the rep
- `Rep` — the salesperson
- `Product` — what was sold (Laptop / Phone / Tablet)
- `Quantity` — how many units
- `Price` — price per unit (in ₹)
- `Status` — where the deal stands: `won`, `pending`, `presented`, `declined`

---

---

# 📌 TOPIC 2 — Categorical Data

> **What it does:** Converts a text column into a special `category` type so you can **control the order** it appears in pivot output.  
> Without this, pandas sorts alphabetically — which may not be the order you want.

In [ ]:
# Right now, 'Status' is just a plain text (object) column
# This means pandas will sort it alphabetically: declined, pending, presented, won
print("Before:", df['Status'].dtype)

In [ ]:
# Convert the 'Status' column to a 'category' data type
df['Status'] = df['Status'].astype('category')

# Now set a CUSTOM ORDER for the categories — the order that makes business sense
# won → pending → presented → declined  (best to worst outcome)
df['Status'] = df['Status'].cat.set_categories(['won', 'pending', 'presented', 'declined'])

# Confirm the dtype has changed from 'object' to 'category'
print("After:", df['Status'].dtype)

# Preview to confirm the data still looks correct
display(df.head())

**Key takeaway:** The data hasn't changed — only how pandas *sorts* this column has changed. This custom order will appear in pivot table output automatically.

---

---

# 📌 TOPIC 3 — Basic Pivot Table (Single Index)

> **What it does:** Summarises your data — like a smarter version of `groupby`.  
> The **minimum** you need is: a DataFrame + an `index`.

The `index` is the column whose unique values become the **row labels** of your pivot.

In [ ]:
# Simplest possible pivot table — just a DataFrame and an index column
# index='Rep' means each unique Rep name becomes a row
# pandas automatically picks numeric columns (Quantity, Price) and calculates their MEAN
pd.pivot_table(df, index='Rep')

In [ ]:
# Change the index to 'Manager' — now each Manager becomes a row
# Notice: only numeric columns (Quantity, Price) appear automatically
pd.pivot_table(df, index='Manager')

**Key takeaway:** Pivot table automatically ignores non-numeric columns and computes the **mean** by default. The `index` decides what the rows represent.

---

---

# 📌 TOPIC 4 — Multiple Indices

> **What it does:** Lets you drill into data at multiple levels — like a hierarchy.  
> Pass a **list** to `index` instead of a single column name.

In [ ]:
# Pass a LIST of columns to index — rows are now grouped by Manager → Rep → Product
# The output shows a multi-level row label (hierarchical index)
pd.pivot_table(df, index=['Manager', 'Rep', 'Product'])

In [ ]:
# Two-level index: Manager then Rep — less granular but cleaner view
# This answers: "What is each Rep's average performance, grouped under their Manager?"
pd.pivot_table(df, index=['Manager', 'Rep'])

**Key takeaway:** The order in the list matters — first item is the outermost group, last item is the most granular.

---

---

# 📌 TOPIC 5 — The `values` Parameter

> **What it does:** Lets you explicitly choose **which columns** to aggregate.  
> Without it, pandas shows all numeric columns. With it, you control exactly what appears.

In [ ]:
# Without values= : both Quantity and Price appear (all numeric columns)
pd.pivot_table(df, index=['Manager', 'Rep'])

In [ ]:
# With values='Price' : only Price is shown — Quantity is excluded
# Use this when you only care about specific columns
pd.pivot_table(df, index=['Manager', 'Rep'], values='Price')

In [ ]:
# Pass a LIST to values to include multiple specific columns
pd.pivot_table(df, index=['Manager', 'Rep'], values=['Price', 'Quantity'])

**Key takeaway:** Always use `values=` in real work — it makes your pivot table explicit and avoids accidental columns appearing.

---

---

# 📌 TOPIC 6 — Aggregation with `aggfunc`

> **What it does:** Controls *how* the values are summarised — sum, mean, count, etc.  
> The default is `mean`. You can pass a single function, a list, or a dictionary.

Build up gradually — start simple, then get powerful.

### 6A — Default (Mean)

In [ ]:
# No aggfunc specified — pandas uses MEAN by default
# This shows the average Price per Manager per Rep
pd.pivot_table(df, index=['Manager', 'Rep'], values='Price')

### 6B — Single Function

In [ ]:
# aggfunc=np.sum — now shows the TOTAL Price instead of the average
# Useful when you want to know total revenue, not average deal size
pd.pivot_table(df, index=['Manager', 'Rep'], values='Price', aggfunc=np.sum)

### 6C — List of Functions

In [ ]:
# Pass a LIST of functions to aggfunc — each function gets its own column
# np.mean gives the average, len gives the COUNT of deals
funcs = [np.mean, len]

pd.pivot_table(df, index=['Manager', 'Rep'], values='Price', aggfunc=funcs)

In [ ]:
# Three functions at once — sum, mean, and standard deviation
# np.std tells you how spread out / variable the prices are
funcs = [np.sum, np.mean, np.std]

pd.pivot_table(df, index=['Manager', 'Rep'], values='Price', aggfunc=funcs)

### 6D — Dictionary of Functions (different function per column)

In [ ]:
# A DICTIONARY lets you apply DIFFERENT functions to DIFFERENT value columns
# len applied to Quantity = count the number of deals
# np.sum applied to Price = total revenue
funcs = {
    'Quantity': len,
    'Price': np.sum
}

pd.pivot_table(df, index=['Manager', 'Rep'], values=['Price', 'Quantity'], aggfunc=funcs)

In [ ]:
# You can even give a LIST of functions to one column in the dictionary
# Quantity: just a count (len)
# Price: both the total (np.sum) AND the average (np.mean)
funcs = {
    'Quantity': len,
    'Price': [np.sum, np.mean]
}

table = pd.pivot_table(df, index=['Manager', 'Status'], values=['Price', 'Quantity'],
                       aggfunc=funcs, fill_value=0)
table

**Key takeaway:** `aggfunc` is where the real power is. Use a list for the same function on all columns, or a dictionary for column-specific control.

---

---

# 📌 TOPIC 7 — The `columns` Parameter

> **What it does:** Spreads the unique values of a column **sideways** as column headers — instead of downward as row labels.

### `columns` vs `values` — the most confusing part for beginners:
| Parameter | Role |
|---|---|
| `values` | The numbers being aggregated (e.g. Price, Quantity) |
| `columns` | A category whose unique values become column headers (e.g. Product) |
| `index` | A category whose unique values become row labels (e.g. Manager, Rep) |

In [ ]:
# columns='Product' — each unique Product (Laptop, Phone, Tablet) becomes a COLUMN
# values='Price' — the number being summarised in each cell is Price
# aggfunc=np.sum — we want total revenue, not average
pd.pivot_table(df, index=['Manager', 'Rep'], values='Price',
               columns='Product', aggfunc=np.sum)

Notice **NaN** values wherever a Rep has no sales for a product. We fix that next.

---

---

# 📌 TOPIC 8 — Handling NaNs with `fill_value`

> **What it does:** Replaces NaN (empty) cells with a value you choose — usually `0`.  
> NaNs appear when a combination of index + column has no matching rows in the data.

In [ ]:
# Same pivot as above — but now fill_value=0 replaces all NaNs with 0
# Much cleaner for reporting — no blank cells
pd.pivot_table(df, index=['Manager', 'Rep'], values='Price',
               columns='Product', aggfunc=np.sum, fill_value=0)

In [ ]:
# fill_value=0 works with multiple values too
# values=['Price', 'Quantity'] — both columns are summarised, NaNs replaced with 0
pd.pivot_table(df, index=['Manager', 'Rep'], values=['Price', 'Quantity'],
               columns='Product', aggfunc=np.sum, fill_value=0)

**Key takeaway:** Always use `fill_value=0` when using `columns=` — it keeps the output clean and avoids confusion.

---

---

# 📌 TOPIC 9 — Moving Items Between Index and Columns

> **What it does:** Shows how the **same data** looks completely different depending on whether a field is in `index` or `columns`.  
> This is a layout choice — neither is wrong. Choose what's clearest for your audience.

In [ ]:
# VERSION A — Product is in columns= (spreads sideways)
# Each product gets its own column, reps are rows
print("Product as COLUMNS:")
display(pd.pivot_table(df, index=['Manager', 'Rep'], values=['Price', 'Quantity'],
                       columns='Product', aggfunc=np.sum, fill_value=0))

In [ ]:
# VERSION B — Product is moved to index= (goes into the rows instead)
# Now each Rep+Product combination is its own row — no columns= at all
print("Product as INDEX (rows):")
display(pd.pivot_table(df, index=['Manager', 'Rep', 'Product'], values=['Price', 'Quantity'],
                       aggfunc=np.sum, fill_value=0))

**Key takeaway:** Same underlying data — completely different visual layout. Use `columns=` when you want a wide/crosstab view. Use `index=` when you want a tall/detailed view.

---

---

# 📌 TOPIC 10 — Margins (Totals)

> **What it does:** Adds a grand total row at the bottom and a grand total column on the right.  
> Set `margins=True` — the totals appear under the label **`All`**.

In [ ]:
# margins=True adds an 'All' row at the bottom (total across all products)
# and an 'All' column on the right (total across all reps)
pd.pivot_table(df, index=['Manager', 'Rep', 'Product'], values=['Price', 'Quantity'],
               aggfunc=[np.sum, np.mean], fill_value=0, margins=True)

**Key takeaway:** `margins=True` is the easiest way to add subtotals and grand totals — great for management reports.

---

---

# 📌 TOPIC 11 — Filtering a Pivot Table

> **What it does:** Once you have a pivot table, it is just a regular DataFrame.  
> You can filter it using `.query()` — the same way you would filter any DataFrame.

We'll use the `table` variable we created earlier (with Manager + Status as index).

In [ ]:
# First, rebuild the pivot table we want to filter
# index: Manager and Status — aggfunc: dict with different functions per column
funcs = {
    'Quantity': len,
    'Price': [np.sum, np.mean]
}

# Store the pivot table in a variable called 'table' so we can filter it below
table = pd.pivot_table(df, index=['Manager', 'Status'], values=['Price', 'Quantity'],
                       aggfunc=funcs, fill_value=0)
table

### 11A — Filter by a Single Value

In [ ]:
# .query() filters the pivot table just like a regular DataFrame filter
# This shows ONLY the rows where Manager is 'Rahul'
# Note: the filter value is wrapped in a list inside the query string
table.query('Manager == ["Rahul"]')

### 11B — Filter by Multiple Values

In [ ]:
# Filter for rows where Status is either 'pending' OR 'won'
# The list ['pending', 'won'] lets you match multiple values at once
table.query('Status == ["pending", "won"]')

**Key takeaway:** A pivot table result is just a DataFrame — all normal DataFrame operations (query, sort, export) work on it.

---

---

# 📌 TOPIC 12 — Styling the Pivot Table

> **What it does:** Adds visual formatting to make patterns in the data immediately obvious.  
> Styling is cosmetic — it doesn't change the data, only how it looks in a notebook.

We'll use the same `table` variable from above.

### 12A — Highlight Max and Min

In [ ]:
# highlight_max() colours the highest value in each column GREEN
# highlight_min() colours the lowest value in each column RED
# Chain both methods together — they apply independently to each column
table.style.highlight_max(color='lightgreen').highlight_min(color='red')

### 12B — Background Gradient

In [ ]:
# background_gradient() applies a colour gradient across each column
# cmap='Blues' means lighter blue = lower value, darker blue = higher value
# At a glance you can spot which cells are high or low
table.style.background_gradient(cmap='Blues')

### 12C — Conditional Formatting with `applymap`

In [ ]:
# Define a custom function that returns a CSS style string for each cell
# v is the cell value — if v > 65000, colour it green, otherwise red
def style_func(v, value, other):
    # Condition: is this cell's value above 65000?
    cond = v > 65000
    # Return the CSS string for 'value' (green) if condition is True, else 'other' (red)
    return value if cond else other

# applymap applies style_func to EVERY cell in the table
# value= is the style when condition is True, other= is the style when False
table.style.applymap(style_func, value='color:green;', other='color:red;')

**Key takeaway:** Use styling to make your reports self-explanatory. Highlight max/min for quick comparison, gradient for a heatmap effect, and `applymap` when you need a custom rule.

---

# ✅ Summary — What We Covered

| # | Topic | Key Parameter / Method | Purpose |
|---|---|---|---|
| 1 | Setup & Data | `pd.read_csv()`, `.head()`, `.dtypes` | Load and inspect data |
| 2 | Categorical Data | `.astype('category')`, `.cat.set_categories()` | Control sort order in pivot output |
| 3 | Basic Pivot Table | `pd.pivot_table(df, index=...)` | Minimum working pivot |
| 4 | Multiple Indices | `index=['Col1', 'Col2']` | Hierarchical row labels |
| 5 | values Parameter | `values='Price'` | Choose which columns to aggregate |
| 6 | aggfunc | `aggfunc=np.sum / [funcs] / {dict}` | Control *how* values are summarised |
| 7 | columns Parameter | `columns='Product'` | Spread a category sideways as headers |
| 8 | fill_value | `fill_value=0` | Replace NaN blanks with 0 |
| 9 | Index vs Columns | Move field between `index=` and `columns=` | Same data, different layout |
| 10 | Margins | `margins=True` | Add grand total row/column |
| 11 | Filtering | `.query('Col == ["val"]')` | Slice the pivot result |
| 12 | Styling | `.style.highlight_max/min/gradient/applymap` | Visual formatting |

---
*End of notebook*